# Monte Carlo Simulation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os
import time

In [2]:
from tools import read_dag_file, add_risk_parameters, simulation

In [3]:
sns.set_theme(palette='tab10')

## Simulations

In [20]:
# Read the original dataset of networks and compute the topological indicators
root, _, files = next(os.walk('data\\set1SP'))
networks = [(file.lower().split('.')[0],os.path.join(root, file)) for file in files if file.endswith('rcp')]

In [5]:
# Risk analysis
# Assume a left-skewed distribution for the risk
# relative to baseline duration
OPT=0.8 # Optimistic
MP=1.1 # Most probable
PE=1.2 # Pessimistic

In [22]:
Ns=10000 # Number of simulation runs
start_time = time.time()
for project_name, project_file in networks:
    print(f'Running simulation for {project_name}')
    # Build the Activity on Node (AON) network
    G=read_dag_file(project_file)
    # Add risk parameters to the network
    add_risk_parameters(G, optim=OPT, mostprob=MP, pessim=PE)
    # Simulate the project
    simulation(G,project_name=project_name,experiment="e1",Nruns=Ns)
print(f"Monte Carlo Simulation took {time.time() - start_time} seconds to run {len(networks)} projects")

Running simulation for ev1
Running simulation for ev10
Running simulation for ev100
Running simulation for ev101
Running simulation for ev102
Running simulation for ev103
Running simulation for ev104
Running simulation for ev105
Running simulation for ev106
Running simulation for ev107
Running simulation for ev108
Running simulation for ev109
Running simulation for ev11
Running simulation for ev110
Running simulation for ev111
Running simulation for ev112
Running simulation for ev113
Running simulation for ev114
Running simulation for ev115
Running simulation for ev116
Running simulation for ev117
Running simulation for ev118
Running simulation for ev119
Running simulation for ev12
Running simulation for ev120
Running simulation for ev121
Running simulation for ev122
Running simulation for ev123
Running simulation for ev124
Running simulation for ev125
Running simulation for ev126
Running simulation for ev127
Running simulation for ev128
Running simulation for ev129
Running simulation 

## Model selection and model evaluation dataset

In [39]:
import random
seed=112358

In [36]:
pd.read_csv(os.path.join(root, files[1]))

,Project,Nodes,Edges,SP,AD,LA,TF,PertDuration
0,ev100,32,61,0.068966,0.666667,0.071895,0.703704,20


In [38]:
dfp

,Project,SP,ModelSelection
0,ev100,0.068966,False
1,ev101,0.172414,False
2,ev102,0.172414,False
3,ev103,0.172414,False
4,ev104,0.172414,False
...,...,...,...
895,ev96,0.068966,False
896,ev97,0.068966,False
897,ev98,0.068966,False
898,ev99,0.068966,False


In [ ]:
# Read the topological indicators files
root, _, files = next(os.walk('montecarlo'))
projects = []
# Read the topological indicators files
for f in files:
    if f.endswith('topological_indicators.csv'):
        df = pd.read_csv(os.path.join(root, f))
        projects.append([df['Project'][0],df['SP'][0]])
dfp=pd.DataFrame(projects, columns=['Project','SP'])
dfp['ModelSelection'] = False
# Select 10 random projects for each SP
for sp in dfp['SP'].unique():
    dfp.loc[dfp[dfp['SP'] == sp].sample(10, random_state=seed).index, 'ModelSelection'] = True
# Save the selected projects to a CSV file
dfp.to_csv('montecarlo\\model_selection_projects.csv', index=False)

In [44]:
# Number of projects selected for each SP
dfp.groupby(['SP','ModelSelection']).count()

Project
SP       ModelSelection         
0.068966 False                90
         True                 10
0.172414 False                90
         True                 10
0.275862 False                90
         True                 10
0.379310 False                90
         True                 10
0.482759 False                90
         True                 10
0.586207 False                90
         True                 10
0.689655 False                90
         True                 10
0.793103 False                90
         True                 10
0.896552 False                90
         True                 10

In [ ]:
# Sample 100 elements from the list
sampled_elements = random.sample(projects, 100)
print(sampled_elements)

## Cross-Validation sets

In [ ]:
# Sample 20% of a simulation runs for testing (the rest will be used for training)
random.seed(seed)
pd.DataFrame(random.sample(range(Ns),int(Ns*0.2)), columns=['test_runs']).to_csv('montecarlo\\test_runs.csv', index=False)